# Clase 5: lectura y escritura de archivos con Python

In [ ]:
#Clasicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
import csv
import json
import pickle
import importlib.util

In [ ]:
# Carpeta de salida.
CARPETA = Path.cwd() / "archivos_clase"
CARPETA.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", Path.cwd())
print("Carpeta de archivos:", CARPETA)

## 0. Guarando figuras

In [ ]:
plt.plot(np.arange(10),np.arange(10))
ruta_figura = CARPETA / "figura.png"
plt.savefig(ruta_figura)

  

## 1.Archivos de texto: .txt (Python estándar)
  Los archivos de texto son legibles sin programas especiales. Puedes usar open(..., mode="w") para escribir, "r" para leer y "a" para agregar contenido.

  Se recomienda with open(...) porque cierra automáticamente el archivo, incluso si se produce un error. Indicamos explícitamente encoding="utf-8".


In [ ]:
ruta_txt = CARPETA / "observaciones.txt"

with open(ruta_txt, "w", encoding="utf-8") as archivo:
    archivo.write("Registro de observaciones astronómicas\n") #¿que hace el \n?
    archivo.write("Objeto: EST-001\n")
    archivo.write("Calidad: buena\n")

with open(ruta_txt, "a", encoding="utf-8") as archivo:
    archivo.write("Comentario: cielo despejado\n")

with open(ruta_txt, "r", encoding="utf-8") as archivo:
    contenido = archivo.read()

contenido



## 2.Alternativa con pathlib

Path.write_text() y Path.read_text() son cómodos para archivos pequeños. Atención: write_text() reemplaza el contenido previo.

In [ ]:
ruta_notas = CARPETA / "notas.txt"
ruta_notas.write_text("Línea 1: adquisición\nLínea 2: calibración\n", encoding="utf-8")
lineas = ruta_notas.read_text(encoding="utf-8").splitlines()
print("Líneas recuperadas:", lineas)
print("¿Existe el archivo?", ruta_notas.exists())

## 3.Archivos CSV:
módulo csv de Python
Un CSV almacena filas y columnas separadas por un delimitador (habitualmente una coma). Es ampliamente compatible, pero no conserva por sí mismo los tipos de datos: al leerlo con csv.DictReader, cada campo llega como cadena de texto (str).

Al abrir archivos CSV con el módulo estándar se recomienda newline="".

In [ ]:
ruta_csv_nativo = CARPETA / "resumen_python.csv"
filas = [{"id": "EST-001", "temperatura_K": 5800, "observada": True},
{"id": "EST-002", "temperatura_K": 7200, "observada": False},
]

with open(ruta_csv_nativo, "w", newline="", encoding="utf-8") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["id", "temperatura_K", "observada"])
    escritor.writeheader()
    escritor.writerows(filas)

with open(ruta_csv_nativo, "r", newline="", encoding="utf-8") as archivo:
    filas_leidas = list(csv.DictReader(archivo))

print(filas_leidas)
print("Tipo del campo temperatura:", type(filas_leidas[0]["temperatura_K"]).__name__)
print("Temperatura convertida:", int(filas_leidas[0]["temperatura_K"]))

## 4.Archivos JSON:

El módulo json de Python JSON es apropiado para configuraciones, diccionarios, listas y metadatos. Es legible por humanos y compatible con otros lenguajes.

Un JSON estándar admite números, cadenas, booleanos, null, listas y objetos. Los arreglos NumPy y los tipos np.float64/np.int64 pueden requerir conversión a tipos de Python, por ejemplo arreglo.tolist() o int(valor).
- json.dump(...): escribe a un archivo.
- json.load(...): lee desde un archivo.
- indent=2: formato legible.
- ensure_ascii=False: conserva caracteres como á, é y ñ.

In [ ]:
ruta_json = CARPETA / "configuracion.json"
configuracion = {
    "proyecto": "Catálogo estelar",
    "instrumento": "Telescopio de práctica",
    "filtros": ["g", "r", "i"],
    "parametros": {"semilla": 2026, "limite_flujo": 1e-11},
    "calibrado": True,
    "comentario": "Medición de estrellas próximas",
}

with open(ruta_json, "w", encoding="utf-8") as archivo:
    json.dump(configuracion, archivo, indent=2, ensure_ascii=False)

with open(ruta_json, "r", encoding="utf-8") as archivo:
    configuracion_leida = json.load(archivo)

assert configuracion_leida == configuracion
print("Filtros:", configuracion_leida["filtros"])
print(ruta_json.read_text(encoding="utf-8")[:300])

## 5.Pickle: guardar objetos de Python


Pickle puede serializar diccionarios, listas, arreglos NumPy y otros objetos de Python sin convertirlos manualmente a texto.

Se usan modos binarios: "wb" (write binary) y "rb" (read binary). El formato no está pensado para inspeccionarse como texto ni para intercambiar datos con otros lenguajes.

**Seguridad: nunca abras con pickle.load() archivos de origen desconocido o no confiable: cargar un Pickle puede ejecutar código. Úsalo solo con archivos que hayas creado tú o una fuente de plena confianza.**

In [ ]:
arreglo = np.array([[1.5, 100.0, 5772.0],[2.3, 150.0, 6200.0],
    [0.8, 200.0, 4500.0], [5.2, 300.0, 7500.0],[1.1, 120.0, 5300.0]])

ruta_pkl = CARPETA / "resultado_python.pkl"
objeto = {
    "nombre": "simulacion",
    "version": 1,
    "matriz": arreglo,
    "indices_validos": np.arange(5),
    "ajustes": {"normalizar": True, "etiqueta": "práctica"},
}

with open(ruta_pkl, "wb") as archivo:
    pickle.dump(objeto, archivo, protocol=pickle.HIGHEST_PROTOCOL)

# SOLO porque el archivo anterior fue generado localmente y es confiable.
with open(ruta_pkl, "rb") as archivo:
    objeto_recuperado = pickle.load(archivo)

assert np.array_equal(objeto["matriz"], objeto_recuperado["matriz"])
print("Claves recuperadas:", list(objeto_recuperado))
print("Forma de matriz recuperada:", objeto_recuperado["matriz"].shape)

In [ ]:
np.array_equal?

## Generamos una tabla sintetica

In [ ]:
rng = np.random.default_rng(2026)
N = 30
luminosidad = rng.uniform(1, 100, N)       # Unidades solares
distancia = rng.uniform(50, 500, N)        # parsec
temperatura = rng.uniform(3000, 10000, N) # kelvin

L_SOL_W = 3.828e26
PC_M = 3.085677581491367e16
flujo = luminosidad * L_SOL_W / (4 * np.pi * (distancia * PC_M) ** 2)

datos = pd.DataFrame({ "id": [f"EST-{i:03d}" for i in range(1, N + 1)], "luminosidad_Lsol": luminosidad, "distancia_pc": distancia, "temperatura_K": temperatura,
    "flujo_W_m2": flujo, "clase": rng.choice(["A", "B", "C"], size=N), })

arreglo = datos[["luminosidad_Lsol", "distancia_pc", "temperatura_K"]].to_numpy()
datos.head()

In [ ]:
 rng.choice(["A", "B", "C"], size=N)

## 6.NumPy: guardar arreglos numéricos

### 6.1 Archivos de texto con np.savetxt() y np.loadtxt()
Útiles para matrices exclusivamente numéricas, fáciles de compartir. El inconveniente es que pueden perder precisión o información de tipo dependiendo del formato elegido. loadtxt espera contenido numérico homogéneo, salvo que se configure de otra manera.

In [ ]:
ruta_array_txt = CARPETA / "propiedades_estelares.txt"
np.savetxt(
    ruta_array_txt,
    arreglo,
    delimiter=",",
    header="luminosidad_Lsol,distancia_pc,temperatura_K",
    comments="# ",
    fmt="%.12e",
)

arreglo_txt = np.loadtxt(ruta_array_txt, delimiter=",", comments="#")
print("Dimensiones leídas:", arreglo_txt.shape)
print("¿Valores próximos a los originales?", np.allclose(arreglo, arreglo_txt))
print("¿Valores iguales a los originales?", np.array_equal(arreglo, arreglo_txt))
print("Primera fila:", arreglo_txt[0])

In [ ]:
np.allclose?

### 6.2 Archivos de texto con np.genfromtxt()
Permite leer archivos de texto que contienen datos numéricos, cadenas de texto y valores ausentes (NaN). A diferencia de np.loadtxt(), ofrece mayor flexibilidad para trabajar con columnas de distintos tipos y datos incompletos. Además, permite identificar las columnas mediante sus nombres y acceder a ellas utilizando arreglos estructurados de NumPy. Es especialmente útil para leer catálogos científicos almacenados en formatos TXT o CSV.

In [ ]:
# Crear un arreglo estructurado con datos numéricos y texto
arreglo = np.array([("Sol", 1.5, 100.0, 5772.0),("Estrella_B", 2.3, 150.0, np.nan),("Estrella_C", np.nan, 200.0, 6500.0),("Estrella_D", 0.8, 300.0, 4300.0)], dtype=[
    ("nombre", "U20"), ("luminosidad_Lsol", "f8"),("distancia_pc", "f8"),("temperatura_K", "f8")])

# Guardar el arreglo en formato CSV
ruta_array_csv = CARPETA / "propiedades_estelares.csv"

np.savetxt(ruta_array_csv,arreglo,delimiter=",",header=",".join(arreglo.dtype.names),comments="",fmt=["%s", "%.12e", "%.12e", "%.12e"])

# Leer el archivo CSV con genfromtxt
arreglo_csv = np.genfromtxt(ruta_array_csv,delimiter=",",names=True,dtype=None,encoding="utf-8" )

print("Dimensiones leídas:", arreglo_csv.shape)
print("Columnas:", arreglo_csv.dtype.names)
print("Tipos de datos:", arreglo_csv.dtype)

# Acceder a columnas individuales
print("Nombres:", arreglo_csv["nombre"])
print("Temperaturas:", arreglo_csv["temperatura_K"])

# Comparar los nombres
print("¿Nombres iguales a los originales?", np.array_equal(arreglo["nombre"],arreglo_csv["nombre"]))

# Comparar las columnas numéricas
for columna in arreglo.dtype.names[1:]:
    print(f"¿{columna} conserva sus valores?",
        np.allclose(arreglo[columna],arreglo_csv[columna],equal_nan=True))

# Mostrar el primer registro
print("Primera estrella:", arreglo_csv[0])

### 6.3 Formato .npy: un arreglo, con su dtype y forma
np.save() y np.load() guardan y recuperan arreglos de forma eficiente. El archivo es binario y requiere software compatible con NumPy.

In [ ]:
arreglo = np.array([[1.5, 100.0, 5772.0],[2.3, 150.0, 6200.0],
    [0.8, 200.0, 4500.0], [5.2, 300.0, 7500.0],[1.1, 120.0, 5300.0]])

ruta_npy = CARPETA / "propiedades_estelares.npy"
np.save(ruta_npy, arreglo)

arreglo_npy = np.load(ruta_npy, allow_pickle=False)
assert np.array_equal(arreglo, arreglo_npy) #¿Que hace assert?
print("Igualdad exacta:", np.array_equal(arreglo, arreglo_npy))
print("Tipo original:", arreglo.dtype, "| Tipo cargado:", arreglo_npy.dtype)

In [ ]:
assert True == False, "True no es False"

### 6.4 Formato .npz: varios arreglos en un archivo
np.savez() guarda varios arreglos; np.savez_compressed() los comprime. Al cargar el .npz, los nombres elegidos funcionan como claves.

Se usa with np.load(...) para cerrar el archivo contenedor correctamente.

In [ ]:
ruta_npz = CARPETA / "catalogo_comprimido.npz"
np.savez_compressed(
    ruta_npz,
    luminosidad=luminosidad,
    distancia=distancia,
    temperatura=temperatura,
    flujo=flujo,
)

with np.load(ruta_npz, allow_pickle=False) as archivo:
    print("Arreglos guardados:", archivo.files)
    luminosidad_recuperada = archivo["luminosidad"].copy()

assert np.array_equal(luminosidad, luminosidad_recuperada)
print("Primeras luminosidades:", luminosidad_recuperada[:4])

## 7.Pandas: guardar y recuperar tablas

### 7.1 CSV: to_csv() y read_csv()
CSV suele ser la primera opción para intercambiar tablas. index=False evita guardar una columna extra.

In [ ]:
ruta_pandas_csv = CARPETA / "catalogo_pandas.csv"
datos.to_csv(ruta_pandas_csv, index=False, encoding="utf-8")

catalogo_csv = pd.read_csv(ruta_pandas_csv)
print("Tamaño:", catalogo_csv.shape)
print("Tipos de datos:")
print(catalogo_csv.dtypes)
display(catalogo_csv.head(3))




In [ ]:
pd.testing.assert_frame_equal(datos, catalogo_csv, check_exact=True, rtol=1e-12)

In [ ]:
print("La tabla recuperada coincide con la original dentro de la tolerancia numérica.")

### 7.2 Filtros y exportación de una selección
Es frecuente crear un archivo nuevo a partir de un subconjunto de la tabla. Aquí seleccionamos estrellas calientes (temperatura_K > 7000).

In [ ]:
calientes = datos.loc[datos["temperatura_K"] > 7000].copy()
ruta_calientes = CARPETA / "estrellas_calientes.csv"
calientes.to_csv(ruta_calientes, index=False)

leidas = pd.read_csv(ruta_calientes)
print("Estrellas seleccionadas:", len(leidas))
assert (leidas["temperatura_K"] > 7000).all()
display(leidas[["id", "temperatura_K", "clase"]].head())

### 7.3 JSON tabular: to_json() y read_json()
orient="records" representa cada fila como un objeto JSON. Es útil para intercambiar datos con servicios web; al recuperar puede ser necesario indicar el tratamiento de fechas o tipos numéricos.

In [ ]:
ruta_pandas_json = CARPETA / "catalogo_pandas.json"
datos.to_json(ruta_pandas_json, orient="records", indent=2,
              force_ascii=False, date_format="iso")

catalogo_json = pd.read_json(ruta_pandas_json, orient="records")
print("Dimensiones:", catalogo_json.shape)
display(catalogo_json.head(2))

### 7.4 Pickle con Pandas: to_pickle() y read_pickle()
Guarda directamente un DataFrame, preservando gran parte de sus propiedades.

**Se mantiene la misma advertencia de seguridad: leer Pickle solo de fuentes confiables.**

In [ ]:
ruta_pandas_pkl = CARPETA / "catalogo_pandas.pkl"
datos.to_pickle(ruta_pandas_pkl)

# Archivo generado por este notebook: origen confiable.
catalogo_pickle = pd.read_pickle(ruta_pandas_pkl)
pd.testing.assert_frame_equal(datos, catalogo_pickle)
print("DataFrame recuperado sin cambios:", catalogo_pickle.shape)

### 8.Otros

Finalmente existen otras formas de guardar datos como [fits](https://es.wikipedia.org/wiki/FITS) o [HDF5](https://en.wikipedia.org/wiki/Hierarchical_Data_Format) que se utilizan largamente en astronomia.